# * TOL : Gross Adds & Subbase

In [2]:
import configparser
import datetime as dt
import pandas as pd
import numpy as np
import xlrd
import oracledb
import re

config = configparser.ConfigParser()
config.read('../../my_config.ini')
config.sections()

TDMDBPR_user = config['TDMDBPR']['username']
TDMDBPR_pwd = config['TDMDBPR']['password']
TDMDBPR_db = config['TDMDBPR']['db']
TDMDBPR_host = config['TDMDBPR']['host']
TDMDBPR_port = config['TDMDBPR']['port']

AKPIPRD_user = config['AKPIPRD']['username']
AKPIPRD_pwd = config['AKPIPRD']['password']
AKPIPRD_db = config['AKPIPRD']['db']
AKPIPRD_host = config['AKPIPRD']['host']
AKPIPRD_port = config['AKPIPRD']['port']

curr_dt = dt.datetime.now().date()
str_curr_dt = curr_dt.strftime('%Y%m%d')

In [3]:
''' Input parameter '''

op_dir = 'data'
op_file = f'tol_ga_subs_monthly_{str_curr_dt}'

# v_year = 0
# v_month_start = 0
# v_month_end = 0
# v_date = 20250101

print(f'\nParameter input...\n')
print(f'   -> op_dir: {op_dir}')
print(f'   -> op_vis_file: {op_file}')
# print(f'\n   -> v_year: {v_year}')
# print(f'   -> v_month_start: {v_month_start}')
# print(f'   -> v_month_end: {v_month_end}')
# print(f'\n   -> v_date: {v_date}')


Parameter input...

   -> op_dir: data
   -> op_vis_file: tol_ga_subs_monthly_20260210


## Import Transaction
-> AGG_PERF_NEWCO

In [1]:
''' Execute transaction '''


# Input parameter
v_as_date = 20260131
print(f'\nParameter input...')
print(f'   -> v_as_date: {v_as_date}')

curr_datetime = dt.datetime.now().strftime('%Y-%m-%d, %H:%M:%S')
print(f'\nData as of {curr_datetime}')


# Connect : TDMDBPR
src_dsn = f'{TDMDBPR_user}/{TDMDBPR_pwd}@{TDMDBPR_host}:{TDMDBPR_port}/{TDMDBPR_db}'
src_conn = oracledb.connect(src_dsn)
src_cur = src_conn.cursor()
query = (f"""
    WITH W_VAR AS 
    ( 
        SELECT V_YYYYMMDD, SUBSTR(V_YYYYMMDD,1,6) AS V_YYYYMM
        FROM (SELECT {v_as_date} AS V_YYYYMMDD FROM DUAL) TMP
    ) 

    SELECT /*+PARALLEL(8)*/ *
    FROM (	
        -->> Active Subs (VIS)
        SELECT TM_KEY_MTH, METRIC_CD, METRIC_NAME, AREA_TYPE, AREA_CD, AREA_NAME
            , ACTUAL_AGG_MTH AS ACTUAL, TARGET_AGG_MTH AS TARGET, PPN_TM
        FROM GEOSPCAPPO.AGG_PERF_NEWCO
        WHERE METRIC_CD = 'TB3S020604' --FTTx SubBase
        AND AREA_TYPE IN ('P', 'G', 'H', 'HH')
        AND TM_KEY_DAY = (SELECT V_YYYYMMDD FROM W_VAR)
        
        UNION ALL 
        
        -->> Gross Adds (VIS)
        SELECT TM_KEY_MTH, METRIC_CD, METRIC_NAME, AREA_TYPE, AREA_CD, AREA_NAME
            , ACTUAL_AGG_MTH AS ACTUAL, TARGET_AGG_MTH AS TARGET, PPN_TM
        FROM GEOSPCAPPO.AGG_PERF_NEWCO
        WHERE METRIC_CD IN (
            'TB3S000102CG' --TOL Gross Adds Connected : Consumer - GEO Channel
            , 'TB3S000102D1CG') --TOL Gross Adds Connected : Consumer - GEO Channel (Install Location)
        AND AREA_TYPE IN ('P', 'G', 'H', 'HH')
        AND TM_KEY_DAY = (SELECT V_YYYYMMDD FROM W_VAR)
        
    --	UNION ALL 
    --	
    --	SELECT TM_KEY_MTH, METRIC_CD, METRIC_NAME, AREA_TYPE, AREA_CD, AREA_NAME
    --		, ACTUAL_AGG_MTH AS ACTUAL, TARGET_AGG_MTH AS TARGET, PPN_TM
    --	FROM GEOSPCAPPO.AGG_PERF_NEWCO_CCAA
    --	WHERE METRIC_CD IN (
    --		'TB3S000102CG' --TOL Gross Adds Connected : Consumer - GEO Channel
    --		, 'TB3S000102D1CG') --TOL Gross Adds Connected : Consumer - GEO Channel (Install Location)
    --	AND AREA_TYPE = 'CCAA'
    --	AND TM_KEY_DAY = (SELECT V_YYYYMMDD FROM W_VAR)
        
        UNION ALL 
        
        -->> Gross Adds (CDS)
        SELECT CAST(SUBSTR(TM_KEY_DAY,1,6) AS INT) AS TM_KEY_MTH
            , METRIC_CD, METRIC_NAME, AREA_TYPE, AREA_CD, AREA_DESC
            , SUM(METRIC_VALUE) AS ACTUAL, NULL AS TARGET
            , MAX(LOAD_DATE) LOAD_DATE
        FROM CDSAPPO.DIM_CORP_KPI 
        WHERE METRIC_CD IN (
            'TB3S000102CG' --TOL Gross Adds Connected : Consumer - GEO Channel
            , 'TB3S000102D1CG' --TOL Gross Adds Connected : Consumer - GEO Channel (Install Location)
            )
        AND AREA_TYPE = 'CCAA'
        AND TM_KEY_DAY LIKE (SELECT V_YYYYMM FROM W_VAR)||'%'
        GROUP BY SUBSTR(TM_KEY_DAY,1,6), METRIC_CD, METRIC_NAME, AREA_TYPE, AREA_CD, AREA_DESC
    ) A
    --ORDER BY TM_KEY_MTH, METRIC_CD, AREA_TYPE, AREA_CD
""")


try:
    # Create Dataframe
    src_cur.execute(query)
    rows = src_cur.fetchall()
    src_df = pd.DataFrame.from_records(rows, columns=[x[0] for x in src_cur.description])
    print(f'\nDataFrame: {src_df.shape[0]} rows, {src_df.shape[1]} columns')
    
    src_cur.close()


except oracledb.DatabaseError as e:
    print(f'\nError with Oracle : {e}')


finally:
    src_conn.close()


Parameter input...
   -> v_as_date: 20260131


NameError: name 'dt' is not defined

In [ ]:
# ''' Generate CSV file '''
# src_df.to_csv(f'{op_dir}/{op_file}.csv', index=False, encoding='utf-8')
# print(f'\n   -> Generate "{op_file}.csv" successfully')